In [63]:
import os
import fitz
import numpy as np
import pandas as pd
import requests
import getpass

In [64]:
username=str(input('Type Your Username:\n'))
print()
password=str(getpass.getpass(prompt='Password'))
crdtstr=str(input('Enter created date start in yyyy-mm-dd:\n'))

Type Your Username:
daroga-minto.matthew

Password········


In [65]:
url = "https://api.urjanet.com/auth/login"

payload = {
    "username": username,
    "password": password
}
h = {
    "accept": "application/json",
    "content-type": "application/json"
}

def Api_tkn_request():
    url = "https://api.urjanet.com/auth/login"
    response = requests.post(url, json=payload, headers=h)
    tkn=response.json()['token']
    headers = {
      'Accept': 'application/json',
      'Authorization': 'Bearer '+tkn}
    return headers

In [67]:
headers=Api_tkn_request()

In [68]:
statements=requests.get('https://api.urjanet.com/utility/statements?search=createdDate=ge='+crdtstr+'&size=100', params={}, headers = headers).json()
pg=statements['page']['totalPages']

In [6]:
statementdict=list()
for j in range(0,pg):
    statements=requests.get('https://api.urjanet.com/utility/statements?search=createdDate=ge='+crdtstr+'&page='+str(j)+'&size=100', params={}, headers = headers).json()
    for i in statements['_embedded']['statements']:
        accountnumber=i['accountNumbers'][0]
        intervalstart=i['periodStart']
        intervalend=i['periodEnd']
        createddate=i['createdDate']
        statementdate=i['statementDate']
        statementID=i['entityId']
        sourcelink=i['_links']['source']['href']
        statementdict.append({'accountnumber':accountnumber,'IntervalStart':intervalstart,'IntervalEnd':intervalend,'statementID':statementID,'createdDate':createddate,'statementDate':statementdate,'SourceLink':sourcelink})
st=pd.DataFrame(statementdict)   
st["Normalisedaccnum"]=[i.replace("-","")   for i in st.accountnumber]

In [7]:
logs=pd.read_excel('Logs.xlsx')

In [8]:
acc=pd.read_excel("Daroga_active.xlsx")

In [9]:
for i in range(len(st)):
    if (str(st["Normalisedaccnum"][i]) not in list(acc["accountNumber"]))&(int(st["Normalisedaccnum"][i]) not in list(acc["NormalizedAccountNumber"])):
        print(st["accountnumber"][i])
        print('Removed SID {} not in active account list'.format(st['statementID'][i]))
        st.drop(labels=i,axis=0,inplace=True)

In [10]:
lst=[]
for i in range(0,len(st)):
    filename=str(st['statementID'][i])+'.pdf'
    doc = fitz.open(filename)
    sid=st['statementID'][i]
    ist=st['IntervalStart'][i]
    ied=st['IntervalEnd'][i]
    meternum=CurrCDG=RemCDG=SubCDG=adj=prior=hbca=totalcdg=csr=netcredit=np.NAN
    for page in doc:
        text = page.get_text()
        
        if text.find("Your billing summary as of") !=-1:
            ind=text.find("Your billing summary as of")
            Statementdate=text[ind+len("Your billing summary as of "):text.find("\n",ind)]
            
        if text.find("Account Number:") !=-1:
            ind=text.find("Account Number: ")
            Accnum=text[ind+len("Account Number: "):text.find("\n",ind)]
        elif text.find("Account:") !=-1:
            ind=text.find("Account: ")
            Accnum=text[ind+len("Account: "):text.find(" Prior",ind)]
        Accnum=Accnum.replace("-","")  
        
        if text.find("Electric Meter Detail")!=-1:
            if text.find("Meter #")!=-1:
                ind=text.find("Meter #")
                meternum=text[text.find("0",text.find("Meter #")):text.find("0",text.find("Meter #"))+9]
            
            
        
        #ind=text.find("0",text.find("Meter #"))
        #if text[ind:ind+9].isnumeric():
        #    meternum=text[ind:ind+9]
        #else:
        #    meternum=""
        
        
        if text.find("CDG Credit Summary")!=-1:
            if text.find("Current Month Allocated CDG Credit") !=-1:
                ind=text.find("Current Month Allocated CDG Credit")
                leng=len("Current Month Allocated CDG Credit")
                CurrCDG=text[ind+leng+1:text.find("\n",ind+leng+1)]
        
            if text.find("Remaining CDG Credit") !=-1:
                ind=text.find("Remaining CDG Credit")
                leng=len("Remaining CDG Credit")
                RemCDG=text[ind+leng+1:text.find("\n",ind+leng+1)]
            
            if text.find("CDG Subscription Fee") !=-1:
                ind=text.find("CDG Subscription Fee")
                leng=len("CDG Subscription Fee")
                SubCDG=text[ind+leng+1:text.find("\n",ind+leng+1)]
            
            if text.find("Adjustments") !=-1:
                ind=text.find("Adjustments")
                leng=len("Adjustments")
                adj=text[ind+leng+1:text.find("\n",ind+leng+1)]
                
            if text.find("Prior Credit") !=-1:
                ind=text.find("Prior Credit")
                leng=len("Prior Credit")
                prior=text[ind+leng+1:text.find("\n",ind+leng+1)]
            
            if text.find("Host Bank Credit Allocation") !=-1:
                ind=text.find("Host Bank Credit Allocation")
                leng=len("Host Bank Credit Allocation")
                hbca=text[ind+leng+1:text.find("\n",ind+leng+1)]
            
            if text.find("Total CDG Credit") !=-1:
                ind=text.find("Total CDG Credit")
                leng=len("Total CDG Credit")
                totalcdg=text[ind+leng+1:text.find("\n",ind+leng+1)]
            
            if text.find("CDG Savings Rate") !=-1:
                ind=text.find("CDG Savings Rate")
                leng=len("CDG Savings Rate")
                csr=text[ind+leng+1:text.find("\n",ind+leng+1)]
                
            if text.find("Total CDG Net Credit") !=-1:
                ind=text.find("Total CDG Net Credit")
                leng=len("Total CDG Net Credit")
                netcredit=text[ind+leng+1:text.find("\n",ind+leng+1)]
       
    dic={"Statement ID":sid,"Account Number":Accnum,"Statement Date":Statementdate,"IntervalStart":ist,"IntervalEnd":ied,
         "Meter Number":meternum,"Adjustments":adj,"Current Month Allocated CDG Credit":CurrCDG,
     "Remaining CDG Credit":RemCDG,"CDG Subscription Fee":SubCDG,"Prior Credit":prior,"Host Bank Credit Allocation":hbca,"Total CDG Credit":totalcdg
        ,"CDG Savings Rate":csr,"Total CDG Net Credit":netcredit}
    lst.append(dic)
    print('Processed SID {}'.format(st['statementID'][i]))

C:\Users\Minto Mathew\Desktop\Daroga power\Bills


In [11]:
df=pd.DataFrame(lst)

In [12]:
os.chdir(os.path.dirname(os.getcwd()))

Processed SID 1eeddd74-fa05-d9ec-99f2-c6dfe8801d64
Processed SID 1eed0071-22de-d4dc-9174-dedf2b32b925
Processed SID 1eed0071-16de-dfea-978b-568be60df41f
Processed SID 1eed0070-fafc-d5da-bcc9-7eb1a502b313
Processed SID 1eed0070-eb53-d8fc-9174-dedf2b32b925
Processed SID 1eed0070-c441-d191-978b-568be60df41f
Processed SID 1eed0070-a55d-d657-bcc9-7eb1a502b313
Processed SID 1eed0070-953f-d6fa-bcc9-7eb1a502b313
Processed SID 1eed0070-5d30-dd16-9174-dedf2b32b925
Processed SID 1eed0070-495f-d955-9174-dedf2b32b925
Processed SID 1eed0070-36dc-dd24-9174-dedf2b32b925
Processed SID 1eed006f-f145-de7e-978b-568be60df41f
Processed SID 1eed006f-ed33-d5f1-bcc9-7eb1a502b313
Processed SID 1eed006f-eaf8-d162-9174-dedf2b32b925
Processed SID 1eed006f-ab4f-db45-bcc9-7eb1a502b313
Processed SID 1eed006f-7fab-dd94-978b-568be60df41f
Processed SID 1eed006f-357c-d266-978b-568be60df41f
Processed SID 1eed006f-0421-de4d-bcc9-7eb1a502b313
Processed SID 1eed006e-efcc-dc50-bcc9-7eb1a502b313
Processed SID 1eed006e-dbd2-d07

In [13]:
generateddate=dt.today()
df.to_excel("DarogaPower_{}.xlsx".format(generateddate),index=False)

In [14]:
pd.concat([logs,st],ignore_index=True).to_excel("Logs.xlsx",index=False)

In [16]:
df["Processing date"]=str(dt.today().strftime('%Y-%m-%d %H:%M:%S'))
his=pd.read_excel("Master.xlsx",dtype={"Processing date":object})
pd.concat([his,df],ignore_index=True).to_excel("Master.xlsx",index=False)

In [17]:
st.describe()